# ShareGPT52K prompt diversity: embeddings → dedup → UMAP → HDBSCAN

This notebook loads **RyokoAI/ShareGPT52K** (ShareGPT-style conversations), extracts user prompts, embeds them, removes near-duplicates, clusters them, and visualizes the result.

**Pipeline:**
1. Load dataset
2. Extract prompt text (first user message or all user messages)
3. Embed (SentenceTransformers by default)
4. Near-duplicate removal (cosine threshold)
5. kNN graph → UMAP (2D)
6. HDBSCAN clustering
7. Inspect clusters (medoids + contrastive keywords)


In [ ]:
# If you run this on a fresh environment, you may need:
# !pip install -q datasets sentence-transformers umap-learn hdbscan scikit-learn plotly

import os
import re
import json
import math
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

import umap
import hdbscan

from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import pairwise_distances

import plotly.express as px


## 1) Load ShareGPT52K and extract prompts

Choose whether you want:
- **first_user**: first human message only (fast, good for intent diversity)
- **all_user**: concatenate all human messages (captures multi-step tasks)


In [ ]:
DATASET_ID = "RyokoAI/ShareGPT52K"
SPLIT = "train"
MODE = "first_user"   # 'first_user' or 'all_user'
SAMPLE_SIZE = 20_000  # set None for full dataset (may be slow)
RANDOM_SEED = 42

# Load raw JSON directly to avoid Arrow conversion errors in this dataset
from huggingface_hub import hf_hub_download

_files = ["sg_90k_part1.json", "sg_90k_part2.json"]
_all_convos = []
for _f in _files:
    _path = hf_hub_download(repo_id=DATASET_ID, filename=_f, repo_type="dataset")
    with open(_path) as _fh:
        _all_convos.extend(json.load(_fh))

# Wrap in a lightweight list-of-dicts so downstream code stays the same
ds = _all_convos

rng = np.random.default_rng(RANDOM_SEED)
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(ds):
    idx = rng.choice(len(ds), size=SAMPLE_SIZE, replace=False)
    ds = [ds[i] for i in idx]


def extract_prompt(example, mode: str = "first_user"):
    conv = example.get("conversations") or example.get("conversation")
    if conv is None:
        return None

    # Common ShareGPT field formats:
    # - list of dicts with keys like {'from': 'human'/'assistant', 'value': '...'}
    # - some variants use {'role': 'user', 'content': '...'}

    user_texts = []
    for turn in conv:
        role = (turn.get("from") or turn.get("role") or "").lower()
        text = turn.get("value") or turn.get("content") or ""
        if role in {"human", "user"} and text.strip():
            user_texts.append(text.strip())
            if mode == "first_user":
                break

    if not user_texts:
        return None

    if mode == "first_user":
        return user_texts[0]
    return "\n\n".join(user_texts)

prompts = [extract_prompt(x, MODE) for x in ds]
prompts = [p for p in prompts if p]
prompts = [re.sub(r"\s+", " ", p) for p in prompts]  # normalize whitespace
prompts = [p for p in prompts if len(p) <= 50000]  # filter out extremely long prompts that may cause issues downstream
prompts = [p for p in prompts if len(p) >= 10]  # filter out very short prompts that may be uninformative
prompts = [re.sub(r"[^\x20-\x7E]+", " ", p) for p in prompts] # replace non-ASCII characters with space to reduce noise (optional, adjust as needed)
prompts = [p for p in prompts if sum(c < "\x80" for c in p) / len(p) > 0.1] # filter out prompts that are mostly non-ASCII (likely noise)
prompts = list(set(prompts))  # deduplicate prompts

print(f"Loaded {len(ds):,} rows → extracted {len(prompts):,} prompts")
print("Example prompt:\n", prompts[0][:500])

## 2) Embed prompts (with caching)

We cache embeddings to disk so you can iterate on clustering/visualization quickly.


In [ ]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # strong default
BATCH_SIZE = 64
CACHE_DIR = Path("./cache")
CACHE_DIR.mkdir(exist_ok=True)

# Cache key depends on dataset, mode, sample size, and a hash of prompt text
hasher = hashlib.md5()
hasher.update((DATASET_ID + SPLIT + MODE + str(SAMPLE_SIZE) + EMBED_MODEL).encode("utf-8"))
# small additional salt so changes in prompt text invalidate cache
hasher.update(str(len(prompts)).encode("utf-8"))
cache_path = CACHE_DIR / f"emb_{hasher.hexdigest()}.npy"

if cache_path.exists():
    embeddings = np.load(cache_path)
    print(f"Loaded cached embeddings: {embeddings.shape} from {cache_path}")
else:
    model = SentenceTransformer(EMBED_MODEL)
    embeddings = model.encode(
        prompts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(cache_path, embeddings)
    print(f"Saved embeddings: {embeddings.shape} to {cache_path}")


## 3) Remove near-duplicates (recommended)

ShareGPT-style corpora contain many near-duplicates. This step improves clustering.

We keep one example from each tight neighborhood above a cosine similarity threshold.


In [ ]:
DEDUP_THRESHOLD = 0.90  # cosine similarity
KNN_K = 100               # neighbors to consider

# First, remove exact duplicates
seen = set()
keep_exact = np.ones(len(prompts), dtype=bool)
for i, p in enumerate(prompts):
    if p in seen:
        keep_exact[i] = False
    else:
        seen.add(p)

prompts_exact = [p for p, k in zip(prompts, keep_exact) if k]
embeddings_exact = embeddings[keep_exact]

# Then, remove near-duplicates from the exact-deduplicated set
nn = NearestNeighbors(n_neighbors=KNN_K, metric="cosine")
nn.fit(embeddings_exact)
dists, nbrs = nn.kneighbors(embeddings_exact, return_distance=True)

keep = np.ones(len(prompts_exact), dtype=bool)
for i in range(len(prompts_exact)):
    if not keep[i]:
        continue
    # mark very close neighbors as duplicates (excluding itself at j=0)
    for j in range(1, KNN_K):
        n = nbrs[i, j]
        sim = 1.0 - dists[i, j]
        if sim >= DEDUP_THRESHOLD:
            keep[n] = False

prompts_d = [p for p, k in zip(prompts_exact, keep) if k]
emb_d = embeddings_exact[keep]
print(f"Exact dedup: {len(prompts_exact):,}/{len(prompts):,} prompts ({len(prompts)-len(prompts_exact):,} exact duplicates removed)")
print(f"Near dedup: {len(prompts_d):,}/{len(prompts_exact):,} prompts ({len(prompts_exact)-len(prompts_d):,} near-duplicates removed)")
print(f"Total: kept {len(prompts_d):,}/{len(prompts):,} prompts ({len(prompts)-len(prompts_d):,} removed)")

## 4) UMAP projection + HDBSCAN clustering

UMAP tends to preserve more global structure than t-SNE, and HDBSCAN works well for uneven, long-tail prompt data.


In [ ]:
UMAP_N_NEIGHBORS = 16
UMAP_MIN_DIST = 0.05

reducer = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    # random_state=RANDOM_SEED,
)
xy = reducer.fit_transform(emb_d)

MIN_CLUSTER_SIZE = 16
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    metric="euclidean",   # clustering in UMAP space
)
labels = clusterer.fit_predict(xy)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = int((labels == -1).sum())
print(f"Clusters: {n_clusters} | noise points: {noise:,} ({noise/len(labels):.1%})")


## 5) Visualize

Tip: hover a point to see the prompt. Color indicates cluster id (noise = -1).


In [ ]:
df = pd.DataFrame({
    "x": xy[:, 0],
    "y": xy[:, 1],
    "cluster": labels,
    "prompt": prompts_d,
})

# filter out noise points for visualization (optional, adjust as needed)
df = df[df["cluster"] != -1]

fig = px.scatter(
    df,
    x="x",
    y="y",
    color="cluster",
    title=f"ShareGPT52K prompts — UMAP (n={len(df):,}) + HDBSCAN (clusters={n_clusters}, noise={noise:,})",
    opacity=0.7,
    height=1000,
    width=1400,
    # custom hover data with text wrapping
    hover_data={"prompt": True, "cluster": True, "x": False, "y": False, "prompt_length": df["prompt"].apply(len)}
)
fig.show()


## 6) Inspect clusters: medoids + keywords

This section helps answer: **"How are clusters different?"**

- **Medoid**: the most central prompt in a cluster
- **Contrastive keywords**: terms overrepresented in the cluster compared to the rest


In [ ]:
def cluster_medoid(cluster_id: int):
    idx = np.where(labels == cluster_id)[0]
    if len(idx) == 0:
        return None
    sub = emb_d[idx]
    # cosine distance among cluster members
    d = pairwise_distances(sub, metric="cosine")
    medoid_local = d.sum(axis=1).argmin()
    return idx[medoid_local]

# Build TF-IDF once
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=50_000,
    ngram_range=(1, 2),
)
X = vectorizer.fit_transform(prompts_d)
terms = np.array(vectorizer.get_feature_names_out())


def top_contrastive_terms(cluster_id: int, topk: int = 12):
    in_idx = np.where(labels == cluster_id)[0]
    out_idx = np.where(labels != cluster_id)[0]
    if len(in_idx) < 5:
        return []

    in_mean = np.asarray(X[in_idx].mean(axis=0)).ravel()
    out_mean = np.asarray(X[out_idx].mean(axis=0)).ravel()

    score = in_mean - out_mean
    top = score.argsort()[-topk:][::-1]
    return [(terms[i], float(score[i])) for i in top if score[i] > 0]

# Show a quick summary of the largest clusters
counts = Counter(labels)
clusters_sorted = [c for c, _ in counts.most_common() if c != -1][:15]

rows = []
for c in clusters_sorted:
    midx = cluster_medoid(c)
    rows.append({
        "cluster": c,
        "size": counts[c],
        "medoid_prompt": prompts_d[midx][:200].replace("\n", " ") + ("…" if len(prompts_d[midx]) > 200 else ""),
        "keywords": ", ".join([t for t, _ in top_contrastive_terms(c, topk=8)]),
    })

summary = pd.DataFrame(rows).sort_values("size", ascending=False)
summary

# save to file
summary.to_csv("cluster_summary.csv", index=False)

## 7) Sample prompts from a chosen cluster


In [ ]:
# CLUSTER_TO_VIEW = int(summary.iloc[0]["cluster"]) if len(summary) else 0
CLUSTER_TO_VIEW = 0
N_SAMPLES = 20

idx = np.where(labels == CLUSTER_TO_VIEW)[0]
print(f"Cluster {CLUSTER_TO_VIEW} size: {len(idx)}")

for i in idx[:N_SAMPLES]:
    print("\n---\n", prompts_d[i][:800])


In [ ]:
from ollama import generate

In [ ]:
generate(
    model="granite4:350m-h",
    prompt="What are common themes in user prompts that ask for coding help? Summarize the main types of requests and provide some example prompts for each type.",
).response